# Rayleigh numbers

In [ ]:
import numpy as np
from lucifex.solver import OptionsPETSc
from lucifex.fdm import AB, CN
from lucifex.sim import parallel_run, as_grid_simulation
from lucifex.viz import plot_colormap, plot_line, set_ipynb_variable
from crocodil.dns.system_a import dns_system_a, SYSTEM_A_REFERENCE

STORE = 1
create_sim = dns_system_a(store_delta=STORE)

NX = 100
NY = 100
CELL_TYPE = 'quadrilateral'
COURANT_ADV = 0.75
COURANT_DIFF = 0.75
COURANT_REAC = 0.1
physical = {k: v for k, v in SYSTEM_A_REFERENCE.items() if k != 'Ra'}
Ra_opts = (600.0, 800.0, 1000.0)

n_proc = set_ipynb_variable('N_PROC', 3)
n_stop = set_ipynb_variable('N_STOP', 600)
t_stop = 20.0
dt_init = 1e-6
n_init = 10

simulations = parallel_run(
    create_sim, n_proc, n_stop,
    dt_init=dt_init, n_init=n_init,
    serialize=as_grid_simulation,
)(
    cell=CELL_TYPE,
    scaling='advective',
    **physical,
    D_adv=AB(1)@CN,
    D_diff=AB(1)@CN,
    courant_adv=COURANT_ADV,
    courant_diff=COURANT_DIFF,
    courant_reac=COURANT_REAC,
    c_stabilization=None,
    c_limits=None,
    c_petsc=OptionsPETSc('gmres', 'ilu'),
    flow_petsc=(OptionsPETSc('cg', 'hypre'), None),
    diagnostic=True,
)(
    Ra=Ra_opts,
)